In [1]:
pip install xarray cfgrib pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import xarray as xr
import numpy as np
import flox
import gc 


In [3]:
# Replace 'your_file.grib' with the actual path to your GRIB file
file_path = 'Dataset/evap_total_cloud_2005-2024.grib'

try:
    ds = xr.open_dataset(file_path, engine='cfgrib')
    #print(ds)
except ValueError as e:
    print(f"Error opening the GRIB file with cfgrib: {e}")

skipping variable: paramId==182 shortName='e'
Traceback (most recent call last):
  File "/home/valau/.local/lib/python3.11/site-packages/cfgrib/dataset.py", line 725, in build_dataset_components
    dict_merge(variables, coord_vars)
  File "/home/valau/.local/lib/python3.11/site-packages/cfgrib/dataset.py", line 641, in dict_merge
    raise DatasetBuildError(
cfgrib.dataset.DatasetBuildError: key present and new value is different: key='time' value=Variable(dimensions=('time',), data=array([1104537600, 1104559200, 1104580800, ..., 1735624800, 1735646400,
       1735668000])) new_value=Variable(dimensions=('time',), data=array([1104516000, 1104559200, 1104602400, ..., 1735538400, 1735581600,
       1735624800]))
/home/valau/.local/lib/python3.11/site-packages/cfgrib/xarray_plugin.py:131: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  va

In [4]:
indices = ['time', 'latitude', 'longitude']
data_vars = list(ds.keys())
relevant_vars = indices + data_vars
relevant_vars

['time', 'latitude', 'longitude', 'tcc']

In [5]:
#ds.groupby(coordinates).mean(dim='x')

In [6]:
df = ds[relevant_vars].to_dataframe().reset_index()[relevant_vars]
df

,time,latitude,longitude,tcc
0,2005-01-01 00:00:00,35.0,-100.00,0.929993
1,2005-01-01 00:00:00,35.0,-99.75,0.945007
2,2005-01-01 00:00:00,35.0,-99.50,0.950012
3,2005-01-01 00:00:00,35.0,-99.25,0.947937
4,2005-01-01 00:00:00,35.0,-99.00,0.940430
...,...,...,...,...
121000015,2024-12-31 18:00:00,25.0,-76.00,0.044586
121000016,2024-12-31 18:00:00,25.0,-75.75,0.114502
121000017,2024-12-31 18:00:00,25.0,-75.50,0.167908
121000018,2024-12-31 18:00:00,25.0,-75.25,0.116699


In [7]:
df['time'] = df['time'].dt.floor('D')#[121000000]
df#.head()

,time,latitude,longitude,tcc
0,2005-01-01,35.0,-100.00,0.929993
1,2005-01-01,35.0,-99.75,0.945007
2,2005-01-01,35.0,-99.50,0.950012
3,2005-01-01,35.0,-99.25,0.947937
4,2005-01-01,35.0,-99.00,0.940430
...,...,...,...,...
121000015,2024-12-31,25.0,-76.00,0.044586
121000016,2024-12-31,25.0,-75.75,0.114502
121000017,2024-12-31,25.0,-75.50,0.167908
121000018,2024-12-31,25.0,-75.25,0.116699


In [8]:
df['time'][15768]

Timestamp('2005-01-01 00:00:00')

In [9]:
gc.collect()

20

In [10]:
#df['time'][15687]

In [11]:
downsampled_df = df.groupby(indices, sort=False)[data_vars].mean()#.reset_index()
downsampled_df

tcc
time       latitude longitude          
2005-01-01 35.0     -100.00    0.952499
                    -99.75     0.963448
                    -99.50     0.964699
                    -99.25     0.963554
                    -99.00     0.967293
...                                 ...
2024-12-31 25.0     -76.00     0.225113
                    -75.75     0.240097
                    -75.50     0.250206
                    -75.25     0.243034
                    -75.00     0.221451

[30250005 rows x 1 columns]

In [12]:
smaller_ds = xr.Dataset.from_dataframe(downsampled_df)

In [13]:
smaller_ds.to_netcdf(path='/home/valau/DS3 Winter 2025 Project/downsampled_datasets/s_evap_total_cloud_2005-2024.grib', mode='w')

In [15]:
# Replace 'your_file.grib' with the actual path to your GRIB file
test_file_path = 'downsampled_datasets/s_evap_total_cloud_2005-2024.grib'

try:
    test_ds = xr.open_dataset(test_file_path, engine='cfgrib')
    #print(ds)
except ValueError as e:
    print(f"Error opening the GRIB file with cfgrib: {e}")

Can't create file 'downsampled_datasets/s_evap_total_cloud_2005-2024.grib.5b7b6.idx'
Traceback (most recent call last):
  File "/home/valau/.local/lib/python3.11/site-packages/cfgrib/messages.py", line 274, in itervalues
    yield self.filestream.message_from_file(file, errors=errors)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/valau/.local/lib/python3.11/site-packages/cfgrib/messages.py", line 341, in message_from_file
    return Message.from_file(file, offset, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/valau/.local/lib/python3.11/site-packages/cfgrib/messages.py", line 105, in from_file
    raise EOFError("End of file: %r" % file)
EOFError: End of file: <_io.BufferedReader name='downsampled_datasets/s_evap_total_cloud_2005-2024.grib'>

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/valau/.local/lib/python3.11/site-packages/cfgrib/messages.py", li

EOFError: No valid message found: 'downsampled_datasets/s_evap_total_cloud_2005-2024.grib'

In [6]:
test_df = test_ds.to_dataframe().reset_index()
test_df

,time,latitude,longitude,tcc
0,2005-01-01 00:00:00,35.0,-100.00,0.929993
1,2005-01-01 00:00:00,35.0,-99.75,0.945007
2,2005-01-01 00:00:00,35.0,-99.50,0.950012
3,2005-01-01 00:00:00,35.0,-99.25,0.947937
4,2005-01-01 00:00:00,35.0,-99.00,0.940430
...,...,...,...,...
121000015,2024-12-31 18:00:00,25.0,-76.00,0.044586
121000016,2024-12-31 18:00:00,25.0,-75.75,0.114502
121000017,2024-12-31 18:00:00,25.0,-75.50,0.167908
121000018,2024-12-31 18:00:00,25.0,-75.25,0.116699


#indices = ['date', 'latitude', 'longitude']
df['time'] = pd.to_datetime({
       'year': df['time'].dt.year,
       'month': df['time'].dt.month,
       'day': df['time'].dt.month,
       'hour': 0
   })
#df.drop(columns=['time'], inplace=True)
df.head()#['hour'] = df

#indices = ['date', 'latitude', 'longitude']
df['time'] = df['time'].dt.date# + pd.Timedelta(hours=12)
#df.drop(columns=['time'], inplace=True)
df.head()#['hour'] = df

downsampled_df.set_index(['date','latitude', 'longitude'], inplace=True)
downsampled_df

#indices = ['date', 'latitude', 'longitude']
#df['date'] = df['time'].dt.date + pd.Timedelta(hours=12)
#df.drop(columns=['time'], inplace=True)
df['time'] = pd.to_datetime({
       'year': df['time'].dt.year,
       'month': df['time'].dt.month,
       'day': df['time'].dt.month,
       'hour': 0
   })
#df.head()#['hour'] = df

# Specify the file path
#filepath = 'path/to/your/file.csv'

# Export DataFrame to CSV
#df.to_csv(filepath, index=False)

#for x in ds.keys():
#    print(x)
#ds['tcc']

#df['time'][15475].date()#.head()

#df['time'][15598].hour

#df['time'][:20].dt.date[0]